# Intent Classification Model Fine-tuning

This notebook fine-tunes a transformer-based classification model for intent detection in a WhatsApp booking assistant.

**Task**: Multi-class classification (14 intent classes)  
**Model**: DistilBERT (lightweight, fast)  
**Data**: Bilingual (English + Roman Urdu) booking assistant messages

---

## 1. Setup & Dependencies

In [ ]:
# Install required packages (run once)
# !pip install transformers datasets scikit-learn pandas torch accelerate

: 

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset as HFDataset, DatasetDict

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Paths
DATA_DIR = Path("../data")
DATASET_PATH = DATA_DIR / "intent_dataset_merged.jsonl"
OUTPUT_DIR = Path("./output")
MODEL_SAVE_PATH = OUTPUT_DIR / "intent_classifier"

# Model configuration
# Using distilbert-base-uncased since Roman Urdu uses Latin script
# Alternative: "bert-base-multilingual-cased" for better multilingual support
MODEL_NAME = "distilbert-base-uncased"

# Training hyperparameters
CONFIG = {
    "max_length": 128,           # Max token length (most messages are short)
    "batch_size": 16,            # Adjust based on GPU memory
    "learning_rate": 2e-5,       # Standard for fine-tuning transformers
    "num_epochs": 10,            # With early stopping
    "weight_decay": 0.01,        # L2 regularization
    "warmup_ratio": 0.1,         # Learning rate warmup
    "eval_steps": 50,            # Evaluate every N steps
    "save_steps": 50,            # Save checkpoint every N steps
}

# Intent labels (14 classes)
INTENT_LABELS = [
    "greeting",
    "booking_request",
    "availability_inquiry",
    "service_selection",
    "date_selection",
    "time_selection",
    "price_inquiry",
    "confirmation",
    "cancellation",
    "modification",
    "information",
    "payment_related",
    "name_provided",
    "unknown"
]

# Create label mappings
label2id = {label: idx for idx, label in enumerate(INTENT_LABELS)}
id2label = {idx: label for idx, label in enumerate(INTENT_LABELS)}

print(f"Number of classes: {len(INTENT_LABELS)}")
print(f"Label mapping: {label2id}")

## 3. Load & Explore Data

In [ ]:
def load_jsonl(filepath):
    """Load JSONL file into list of dictionaries."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load the merged dataset
raw_data = load_jsonl(DATASET_PATH)
print(f"Total samples loaded: {len(raw_data)}")

# Convert to DataFrame for easier exploration
df = pd.DataFrame(raw_data)
df.head(10)

In [ ]:
# Dataset statistics
print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)

print(f"\nTotal samples: {len(df)}")
print(f"Unique intents: {df['intent'].nunique()}")

# Intent distribution
intent_counts = df['intent'].value_counts()
print("\nIntent Distribution:")
print("-" * 40)
for intent, count in intent_counts.items():
    pct = (count / len(df)) * 100
    bar = "█" * int(pct / 2)
    print(f"{intent:22s}: {count:4d} ({pct:5.1f}%) {bar}")

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(12, 6))
intent_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Intent Class Distribution', fontsize=14)
ax.set_xlabel('Intent', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=45)

# Add count labels on bars
for i, (intent, count) in enumerate(intent_counts.items()):
    ax.text(i, count + 2, str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Text Length Statistics:")
print(df[['text_length', 'word_count']].describe())

# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['text_length'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=20, color='coral', edgecolor='black')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Sample examples from each class
print("Sample Examples by Intent:")
print("=" * 60)

for intent in INTENT_LABELS:
    samples = df[df['intent'] == intent]['text'].head(3).tolist()
    print(f"\n{intent.upper()}:")
    for s in samples:
        print(f"  • {s}")

## 4. Data Preprocessing

In [ ]:
# Prepare data: text and numeric labels
texts = df['text'].tolist()
labels = [label2id[intent] for intent in df['intent'].tolist()]

print(f"Total samples: {len(texts)}")
print(f"Label range: {min(labels)} - {max(labels)}")

In [ ]:
# Split into train/validation/test (80/10/10)
# First split: 80% train, 20% temp
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

# Second split: 50% of temp for validation, 50% for test (10% each of total)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
)

print(f"Train set: {len(train_texts)} samples")
print(f"Validation set: {len(val_texts)} samples")
print(f"Test set: {len(test_texts)} samples")

# Verify stratification
print("\nLabel distribution in splits:")
for name, lbls in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    dist = Counter(lbls)
    print(f"  {name}: {len(dist)} classes represented")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Test tokenization on sample texts
sample_texts = [
    "Hi",
    "Aoa padel slot book karna hai kal shaam",
    "slot hai? evening me?",
    "7pm"
]

print("Tokenization Examples:")
print("-" * 50)
for text in sample_texts:
    tokens = tokenizer.tokenize(text)
    print(f"Text: '{text}'")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {tokenizer.encode(text)}")
    print()

In [ ]:
# Create HuggingFace datasets
def create_hf_dataset(texts, labels):
    """Create a HuggingFace Dataset from texts and labels."""
    return HFDataset.from_dict({
        "text": texts,
        "label": labels
    })

# Create dataset dict
dataset = DatasetDict({
    "train": create_hf_dataset(train_texts, train_labels),
    "validation": create_hf_dataset(val_texts, val_labels),
    "test": create_hf_dataset(test_texts, test_labels)
})

print(dataset)

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    """Tokenize texts with padding and truncation."""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_length"]
    )

# Apply tokenization
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]  # Remove original text column
)

print("Tokenized dataset:")
print(tokenized_dataset)
print(f"\nFeatures: {tokenized_dataset['train'].features}")

## 5. Model Setup

In [ ]:
# Load pre-trained model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(INTENT_LABELS),
    id2label=id2label,
    label2id=label2id
)

print(f"Model: {MODEL_NAME}")
print(f"Number of labels: {model.config.num_labels}")
print(f"Total parameters: {model.num_parameters():,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Define metrics computation function
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, F1 for evaluation."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## 6. Training

In [ ]:
# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training arguments
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    
    # Training settings
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"] * 2,
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    
    # Evaluation settings
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    
    # Best model selection
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Logging
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=25,
    report_to="none",  # Disable wandb/tensorboard
    
    # Performance
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    dataloader_num_workers=0,  # Set to 0 for Windows compatibility
    
    # Reproducibility
    seed=SEED,
)

print("Training Arguments:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Trainer initialized successfully!")

In [ ]:
# Train the model
print("Starting training...")
print("=" * 50)

train_result = trainer.train()

print("\nTraining completed!")
print(f"Total training time: {train_result.metrics['train_runtime']:.2f} seconds")

In [ ]:
# Plot training history
history = trainer.state.log_history

# Extract training and validation losses
train_losses = [(h['step'], h['loss']) for h in history if 'loss' in h and 'eval_loss' not in h]
eval_data = [(h['step'], h['eval_loss'], h['eval_f1']) for h in history if 'eval_loss' in h]

if train_losses and eval_data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss plot
    train_steps, train_loss_vals = zip(*train_losses)
    eval_steps, eval_loss_vals, eval_f1_vals = zip(*eval_data)
    
    axes[0].plot(train_steps, train_loss_vals, label='Train Loss', color='blue', alpha=0.7)
    axes[0].plot(eval_steps, eval_loss_vals, label='Val Loss', color='red', marker='o')
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # F1 Score plot
    axes[1].plot(eval_steps, eval_f1_vals, label='Val F1', color='green', marker='o')
    axes[1].set_xlabel('Steps')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_title('Validation F1 Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Not enough training history to plot.")

## 7. Evaluation

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset["test"])

print("\n" + "=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Get predictions for detailed analysis
predictions_output = trainer.predict(tokenized_dataset["test"])
predictions = np.argmax(predictions_output.predictions, axis=1)
true_labels = test_labels

# Classification report
print("\nDetailed Classification Report:")
print("=" * 70)
print(classification_report(
    true_labels, 
    predictions, 
    target_names=INTENT_LABELS,
    zero_division=0
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=INTENT_LABELS,
    yticklabels=INTENT_LABELS
)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title('Confusion Matrix - Intent Classification', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze misclassifications
print("\nMisclassification Analysis:")
print("=" * 70)

misclassified = []
for i, (text, true_label, pred_label) in enumerate(zip(test_texts, true_labels, predictions)):
    if true_label != pred_label:
        misclassified.append({
            "text": text,
            "true": id2label[true_label],
            "predicted": id2label[pred_label]
        })

print(f"Total misclassified: {len(misclassified)} / {len(test_texts)} ({100*len(misclassified)/len(test_texts):.1f}%)")
print("\nSample Misclassifications:")
print("-" * 70)

for item in misclassified[:10]:  # Show first 10
    print(f"Text: '{item['text']}'")
    print(f"  True: {item['true']} | Predicted: {item['predicted']}")
    print()

## 8. Save Model

In [ ]:
# Save the final model
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_SAVE_PATH))
tokenizer.save_pretrained(str(MODEL_SAVE_PATH))

print(f"Model saved to: {MODEL_SAVE_PATH}")
print(f"\nSaved files:")
for f in MODEL_SAVE_PATH.iterdir():
    print(f"  - {f.name}")

In [ ]:
# Save label mappings
import json

label_config = {
    "labels": INTENT_LABELS,
    "label2id": label2id,
    "id2label": id2label
}

with open(MODEL_SAVE_PATH / "label_config.json", "w") as f:
    json.dump(label_config, f, indent=2)

print("Label configuration saved!")

## 9. Inference Example

In [ ]:
# Load saved model for inference
from transformers import pipeline

# Create classification pipeline
classifier = pipeline(
    "text-classification",
    model=str(MODEL_SAVE_PATH),
    tokenizer=str(MODEL_SAVE_PATH),
    device=0 if torch.cuda.is_available() else -1
)

print("Inference pipeline created!")

In [ ]:
# Test inference
test_messages = [
    "Aoa",
    "slot book karna hai kal shaam",
    "koi slot available hai?",
    "padel court",
    "tomorrow",
    "7pm",
    "kitna hai?",
    "han book kardo",
    "cancel karo",
    "time change kardo",
    "parking hai?",
    "jazzcash se payment",
    "mera naam Ali hai",
    "asdfgh"
]

print("Inference Results:")
print("=" * 60)

for message in test_messages:
    result = classifier(message)[0]
    print(f"'{message}'")
    print(f"  → {result['label']} (confidence: {result['score']:.3f})")
    print()

In [ ]:
# Function for easy inference in production
def predict_intent(text: str, return_confidence: bool = True):
    """
    Predict the intent of a given text message.
    
    Args:
        text: Input message
        return_confidence: Whether to return confidence score
    
    Returns:
        Intent label (and confidence if requested)
    """
    result = classifier(text)[0]
    
    if return_confidence:
        return result['label'], result['score']
    return result['label']

# Test the function
intent, confidence = predict_intent("Aoa bhai padel slot book karna hai kal shaam 6 bajay")
print(f"Intent: {intent}")
print(f"Confidence: {confidence:.3f}")

## 10. Summary

### Training Complete!

This notebook trained a DistilBERT-based intent classifier with:
- **14 intent classes** for a booking assistant
- **Bilingual support** (English + Roman Urdu)
- **~700 training samples**

### Next Steps:
1. **Improve performance**: Add more training data, especially for underrepresented classes
2. **Try other models**: `bert-base-multilingual-cased` for better multilingual support
3. **Deploy**: Use the saved model in your backend API
4. **Monitor**: Track real-world performance and retrain periodically

### Model Files:
- Saved to: `./output/intent_classifier/`
- Ready for use with HuggingFace `pipeline()` or direct loading